# Multi-cue ablation (additive to frozen Results)

**Does not change** Tables 1–11, Appendix C, or `main-output/`.
The published Multi-cue remains **0.968 ± 0.011** (Table 2).

This notebook (1) rebuilds the ablation summary from frozen Table 2 + T10,
(2) maps each switch onto the video fusion taxonomy, and (3) optionally runs
video ablations that write **only** under `outputs/ablation_multicue/`.


In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.switch_backend("Agg")

ROOT = Path("..").resolve()
MAIN = ROOT / "main-output"
OUT = ROOT / "outputs" / "ablation_multicue"
TAB = ROOT / "f2-result" / "tables"
FIG = Path("figures")
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

VARIANT_ORDER = [
    "cue_appearance", "cue_context", "concat",
    "attention_only", "gating_only", "full",
]
print("ROOT:", ROOT)
assert MAIN.exists(), "main-output missing"
# Fingerprint check: summary must still be the frozen Multi-cue row
s = pd.read_csv(MAIN / "summary.csv")
mc = s.loc[s["model"] == "Multi-cue"].iloc[0]
print(f"Frozen Multi-cue mean={mc['mean_validation_accuracy']:.6f} (expect ~0.9676)")


## 1. Rebuild ablation summary from frozen artifacts (no retrain)

Reads `main-output/summary.csv` + `T10_linear_probe.csv` only. Writes T12 under
`outputs/ablation_multicue/` and `f2-result/tables/`.


In [ ]:
cmd = [sys.executable, str(ROOT / "f2-result" / "code" / "build_ablation_summary_from_existing.py")]
subprocess.run(cmd, cwd=str(ROOT), check=True)
summary = pd.read_csv(OUT / "ablation_summary.csv")
# Prove main-output was not rewritten by comparing to the in-memory freeze
s2 = pd.read_csv(MAIN / "summary.csv")
assert s2.equals(s), "main-output/summary.csv changed — abort"
summary


## 2. Results summary (maps onto published Multi-cue)

| Ablation | Meaning on published MultiCueNet | Number |
|---|---|---|
| (i) appearance alone | first 24-d block | 0.943 (T10) |
| (ii) concat / ungated 40-d | both blocks, no deep gate | 0.965 (T10) |
| (iv)/(v) full gated model | Table 2 Multi-cue | **0.968 ± 0.011** |
| (iii) attention only | N/A on stand-in | video pipeline only |

Same conclusion as Table 10: most signal is in the 24-d block; the gate adds the last ~2 points over that probe.


In [ ]:
pub = pd.read_csv(OUT / "ablation_table_manuscript.csv")
print(pub.to_string(index=False))

# Δ plot for filled rows
plot = summary.dropna(subset=["mean_accuracy"]).copy()
plot = plot[plot["variant"].isin(VARIANT_ORDER)]
fig, ax = plt.subplots(figsize=(7.5, 4))
x = np.arange(len(plot))
y = plot["mean_accuracy"].to_numpy()
yerr = plot["sd_accuracy"].fillna(0).to_numpy()
colors = ["#0072B2" if v == "full" else "#56B4E9" for v in plot["variant"]]
ax.bar(x, y, yerr=yerr, color=colors, alpha=0.9, capsize=4, width=0.7)
ax.axhline(0.967582, color="#D55E00", ls="--", lw=1.2, label="Table 2 Multi-cue (frozen)")
ax.set_xticks(x)
ax.set_xticklabels(plot["description"], rotation=25, ha="right")
ax.set_ylabel("Accuracy")
ax.set_ylim(0.90, 1.01)
ax.legend(frameon=False)
ax.set_title("Multi-cue ablation anchored on frozen Results")
fig.tight_layout()
fig_path = FIG / "multicue_ablation_vs_table2.png"
fig.savefig(fig_path, dpi=200)
plt.close(fig)
print("Wrote", fig_path)


## 3. Video ablation (optional; additive only)

Does **not** replace Table 2. Set `RUN_VIDEO_ABLATION = True` to train motion/face/hands
variants; outputs land only in `outputs/ablation_multicue/<variant>/`.


In [ ]:
RUN_VIDEO_ABLATION = False
SMOKE_MAX_VIDEOS = 0
if RUN_VIDEO_ABLATION:
    cmd = [
        sys.executable, str(ROOT / "scripts" / "run_multimodal_ablation.py"),
        "--results-dir", str(OUT / "video_run"),
        "--feature-cache", str(ROOT / "data" / "cache" / "multimodal_features.npz"),
    ]
    if SMOKE_MAX_VIDEOS:
        cmd += ["--max-videos", str(SMOKE_MAX_VIDEOS)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
else:
    print("Skipping video retrain (RUN_VIDEO_ABLATION=False). Frozen T12 summary above stands.")


## 4. Architecture check (video model graphs; random tensors)

Confirms each video ablation variant compiles. No writes to `main-output/`.


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
sys.path.insert(0, str(ROOT))
from models.multimodal import Multimodal, ABLATION_VARIANTS

cfg = dict(num_classes=5, max_frames=20, motion_dim=8, facial_dim=15, gesture_dim=45,
           attention_heads=4, fusion_units=64, dropout_rate=0.3, learning_rate=5e-4)
B, T = 4, 20
Xm = np.random.randn(B, T, 8).astype("float32")
Xf = np.random.randn(B, T, 15).astype("float32")
Xh = np.random.randn(B, T, 45).astype("float32")
rows = []
for v in ABLATION_VARIANTS:
    m = Multimodal({**cfg, "ablation_variant": v})
    Xin = Multimodal.select_inputs_for_variant([Xm, Xf, Xh], v)
    out = m.model(Xin, training=False)
    rows.append({"variant": v, "out": tuple(out.shape), "params": int(m.model.count_params())})
    m.clear_session()
pd.DataFrame(rows)
